Utils and data_io Preprocessing

In [2]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import re

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
print(REPO_ROOT)

c:\Users\bng\Projects\peak_detection\peak_detection


In [3]:
from utils import simplify_label

In [4]:
def label_parsing(row):
    element_set = set()
    range_pattern = re.compile(r'Range(\d+)=([\d.]+) ([\d.]+) (.*) Color:([0-9A-Fa-f]{6})')
    match = range_pattern.match(row.strip())
    raw_label = match.group(4).strip()
    print(raw_label)
    label = re.sub(r'Vol:[\d.]+', '', raw_label).strip()
    species_parts = re.findall(r'\b([A-Z][a-z]?):(\d+)\b', label)
    print(species_parts)
    if not species_parts: 
    # just because the label is not in RRNG format, we can call it unknown and retain for RangingNN evaluation
        label_simple = "Species Unknown"
    else:
        print(f'species_parts: {species_parts}')
        for sym, _ in species_parts:
            element_set.add(sym)
        label = " ".join(f"{sym}:{count}" for sym, count in species_parts)
        print(f'label: {label}')
        label_simple = simplify_label(label)
        print(f'label_simple: {label_simple}')
    print(label_simple)
    print(element_set)

In [5]:
example_rrng = [
    "Range95=0.9950 1.0970 Vol:0.00000 Name:69 Color:CCCC00",
    "Range96=2.0020 2.0760 Vol:0.00000 Name:da18.5 Color:008000",
    "Range97=3.0100 3.0390 Vol:0.00000 Name:IFB Color:0000FF",
    "Range98=16.9600 17.0230 Vol:0.02883 O:1 H:1 Color:009999",
    "Range99=17.9850 18.0520 Vol:0.02883 H:2 O:1 Color:CCCC00",
]

for range in example_rrng:
    label_parsing(range)

Vol:0.00000 Name:69
[]
Species Unknown
set()
Vol:0.00000 Name:da18.5
[]
Species Unknown
set()
Vol:0.00000 Name:IFB
[]
Species Unknown
set()
Vol:0.02883 O:1 H:1
[('O', '1'), ('H', '1')]
species_parts: [('O', '1'), ('H', '1')]
label: O:1 H:1
label_simple: HO
HO
{'H', 'O'}
Vol:0.02883 H:2 O:1
[('H', '2'), ('O', '1')]
species_parts: [('H', '2'), ('O', '1')]
label: H:2 O:1
label_simple: H2O
H2O
{'H', 'O'}


In [6]:
test_data = r"C:\Users\bng\Projects\peak_detection\data\RRNG_test\R13_40310Zr Top Level ROI.RRNG"
from peak_detection.data_io import parse_rrng

print(parse_rrng(test_data))

([PeakRange(start=0.995, end=1.4, pos=1.1975, label='H', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=1.987, end=2.162, pos=2.0745, label='H2', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=13.407, end=17.727, pos=15.567, label='Al', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=26.882, end=27.932, pos=27.407, label='Al', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=22.905, end=23.235, pos=23.07, label='Ti', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=23.395, end=23.765, pos=23.58, label='Ti', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=23.882, end=24.331, pos=24.1065, label='Ti', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=24.382, end=24.837, pos=24.6095, label='Ti', id_score=0.0, method='', detailed_id=None, is_unknown=False), PeakRange(start=24.905, end=2

In [15]:
path = r"C:\Users\bng\Projects\peak_detection\peak_detection\IonIdentificationModels\training_data\NewData_truthcoverage_lightmol1p_C3_BO_C2O_2p_2026-06-10\Data0001"
file = "000000.csv"

threshold_c=1e-8

def min_max_scale(ar):
    """Min-max normalize an array to [0, 1]."""
    return (ar - ar.min()) / (ar.max() - ar.min())

df = pd.read_csv(os.path.join(path, file), keep_default_na=False)
mc = df.get(['mc']).to_numpy().squeeze()
counts = df.get(['counts']).to_numpy().squeeze()

if counts.max() == counts.min():
    counts = np.zeros_like(counts)
else:
    counts = min_max_scale(counts)

indexes = counts > threshold_c

ions_raw = df.get(['ion']).to_numpy().squeeze()
ions2_raw = df.get(['ion2']).to_numpy().squeeze()

target_ions = []
for i1, i2 in zip(ions_raw, ions2_raw):
    if i2 and i2 != "":
        target_ions.append(i2)
    else:
        target_ions.append(i1)
ions = np.array(target_ions)

mc_f = mc[indexes]
ions_f = ions[indexes]

print(mc_f)
print(ions_f)

[  1.02         1.98         2.97         5.98         6.49
   6.99         7.47         7.97         9.03        10.98
  12.          12.96        14.02        14.61666667  14.65666667
  15.03        15.97        16.96        17.55        17.99
  18.03        18.49        18.99        19.96        21.99
  23.98        24.95        25.95        26.96        28.03
  28.95        29.02        31.96        33.05        33.95
  34.03        34.95        35.5         35.96        36.02
  36.53        36.98        40.          44.          47.31333333
  47.97        48.02        48.66666667  71.02        71.99
  72.04        73.02       106.95       109.         141.98
 143.99       144.02       145.98      ]
['H' 'H2' 'H3' 'C' 'C' 'N' 'N' 'O' 'O' 'CO2' 'OOO' 'C' 'N' 'CO2' 'N2O' 'N'
 'OOO' 'HO' 'Cl' 'H2O' 'C3' 'Cl' 'H3O' 'C2O' 'CO2' 'OOO' 'C2' 'CN' 'CN'
 'N2' 'N2' 'N2' 'O2' 'HO2' 'HHOO' 'O2' 'Cl' 'AgCl' 'AgCl' 'C3' 'AgCl' 'Cl'
 'C2O' 'CO2' 'AgCl' 'OOO' 'AgCl' 'AgCl' 'AgCl' 'AgCl' 'AgCl' 'AgC